# Preparation notebook
**raw Excel -> cleanded long dataset -> processed CSV**

In [1]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# Establish path
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from config.project_paths import (
    ROOT,
    DATA_RAW,
    DATA_PROCESSED,
    FIGURES,
    YEAR_MIN,
    YEAR_MAX,
    get_raw_data_path,
    get_processed_data_path,
    get_figure_path,
    validate_paths,
)

In [2]:
# Verify all paths
print("ROOT:", ROOT)
print("RAW DATA:", DATA_RAW)
print("PROCESSED DATA:", DATA_PROCESSED)
print("FIGURES:", FIGURES)

ROOT: /home/ggirelli/Documents/Learning_resources/DataAnalysis/Projects/Capstone_1
RAW DATA: /home/ggirelli/Documents/Learning_resources/DataAnalysis/Projects/Capstone_1/data/raw
PROCESSED DATA: /home/ggirelli/Documents/Learning_resources/DataAnalysis/Projects/Capstone_1/data/processed
FIGURES: /home/ggirelli/Documents/Learning_resources/DataAnalysis/Projects/Capstone_1/outputs/figures


In [3]:
# Load raw data
RAW_DEMOGRAPHIC_ASPECTS_FILE = DATA_RAW / "Demographic-aspects-2023.xlsx"

In [4]:
# Create "if statement" to test for errors
if not RAW_DEMOGRAPHIC_ASPECTS_FILE.exists():
    raise FileNotFoundError

In [5]:
# Set variable for wide format
df = pd.read_excel(RAW_DEMOGRAPHIC_ASPECTS_FILE, header=1)

In [6]:
df

,Key Demographic aspects,Unit,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Males,Absolute,51309.0,51462.0,51399.0,51512.0,51515.0,50930.0,50664.0,50440.0,50559.0
1,Females,Absolute,57326.0,57357.0,57253.0,57652.0,57726.0,57002.0,56804.0,56712.0,57007.0
2,Total population,Absolute,108635.0,108818.0,108651.0,109164.0,109241.0,107932.0,107468.0,107152.0,107566.0
3,Sex ratio,NaN,89.5,89.7,89.8,89.3,89.2,89.3,89.2,88.9,88.7
4,Pop. per km2 land,Absolute,604.0,605.0,604.0,606.0,607.0,600.0,597.0,595.0,598.0
5,Pop. 14 years and younger,%,18.9,18.7,18.6,18.3,18.1,17.0,16.6,16.1,15.6
6,Pop. 60 years and older,%,18.7,19.4,20.1,20.9,21.7,23.5,24.5,25.5,26.3
7,Average age,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Males,Years,37.5,37.8,38.1,38.4,38.8,39.8,40.2,40.6,41.0
9,Females,Years,40.1,40.4,40.7,41.1,41.5,42.4,42.9,43.4,43.8


In [7]:
filtered = df[
df["Key Demographic aspects"].isin(["Males", "Females"])]

filtered

,Key Demographic aspects,Unit,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Males,Absolute,51309.0,51462.0,51399.0,51512.0,51515.0,50930.0,50664.0,50440.0,50559.0
1,Females,Absolute,57326.0,57357.0,57253.0,57652.0,57726.0,57002.0,56804.0,56712.0,57007.0


In [8]:
filtered = filtered.drop(columns=["Unit"])

In [9]:
filtered

,Key Demographic aspects,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Males,51309.0,51462.0,51399.0,51512.0,51515.0,50930.0,50664.0,50440.0,50559.0
1,Females,57326.0,57357.0,57253.0,57652.0,57726.0,57002.0,56804.0,56712.0,57007.0


In [10]:
long_df = filtered.melt(
    id_vars = ["Key Demographic aspects"],
    var_name = "Sex",
    value_name = "Population"
)

In [11]:
long_df = long_df.rename(columns={"Key Demographic aspects" : "sex", "Sex" : "year", "Population" : "population"})

In [12]:
long_df = long_df.sort_values(by="sex", ascending=[True])

In [13]:
long_df.reset_index(drop=True)

,sex,year,population
0,Females,2015,57326.0
1,Females,2016,57357.0
2,Females,2017,57253.0
3,Females,2018,57652.0
4,Females,2019,57726.0
5,Females,2020,57002.0
6,Females,2021,56804.0
7,Females,2022,56712.0
8,Females,2023,57007.0
9,Males,2015,51309.0


In [14]:
long_df["annual_change"] = long_df.groupby("sex")["population"].diff()

In [15]:
long_df.reset_index(drop=True)

,sex,year,population,annual_change
0,Females,2015,57326.0,NaN
1,Females,2016,57357.0,31.0
2,Females,2017,57253.0,-104.0
3,Females,2018,57652.0,399.0
4,Females,2019,57726.0,74.0
5,Females,2020,57002.0,-724.0
6,Females,2021,56804.0,-198.0
7,Females,2022,56712.0,-92.0
8,Females,2023,57007.0,295.0
9,Males,2015,51309.0,NaN


In [16]:
long_df.to_csv("population_change_by_sex.csv", index=False)